# Chapter 2 (leptonic) — Notebook 4: Cut optimisation

**Goals**

- Scan a cut threshold and compute the **weighted** significance $S/\sqrt{S+B}$.

In [ ]:
%matplotlib inline
import awkward as ak
import numpy as np
import matplotlib.pyplot as plt

from topmass import io, kinematics, selection, plotting, fitting, neutrino, pairing, weights, style
from topmass.constants import M_W, M_TOP

In [ ]:
io.setup()
samples = io.build_samples()
procs = ['ttbar', 'single_top', 'diboson']
evs = {p: io.load_process(p, samples, fraction=0.1) for p in procs}

thresholds = np.arange(30, 130, 10)
sig = []
for t in thresholds:
    cuts = selection.SemilepCuts(met_min=float(t))
    yields = {}
    for p, ev in evs.items():
        sel = selection.semilep_preselection(ev, cuts)
        yields[p] = float(ak.sum(weights.weight_for(p, ev)[sel]))
    s = yields['ttbar']
    b = yields['single_top'] + yields['diboson']
    sig.append(plotting.significance([s], [b]))

plt.plot(thresholds, sig, 'o-')
plt.xlabel('MET cut [GeV]'); plt.ylabel(r'$S/\sqrt{S+B}$')

## ✏️ Your turn 4.1

▶️ Change the values being scanned and re-run.

This scans the jet-count and b-jet-count requirements and prints the weighted significance
$S/\sqrt{S+B}$ for each working point. Which `(n_jets_min, n_bjets_min)` combination maximises it?

> **Stretch (optional):** add a third value to `NB_VALUES` (e.g. `3`).

In [ ]:
NJ_VALUES = (4, 5, 6)    # ✏️ try (3, 4, 5)
NB_VALUES = (1, 2)       # ✏️ try (1, 2, 3)

print(f'{"n_jets":>7} {"n_bjets":>8} {"S/sqrt(S+B)":>13}')
for nj in NJ_VALUES:
    for nb in NB_VALUES:
        cuts = selection.SemilepCuts(n_jets_min=nj, n_bjets_min=nb)
        y = {p: float(ak.sum(weights.weight_for(p, ev)[selection.semilep_preselection(ev, cuts)]))
             for p, ev in evs.items()}
        s, b = y['ttbar'], y['single_top'] + y['diboson']
        print(f'{nj:>7} {nb:>8} {plotting.significance([s], [b]):>13.2f}')